# Customer Lifetime Value — Non-Contractual Model (BG/NBD + Gamma-Gamma)
### Predicting Future Purchases and CLV in a Non-Subscription Business

## 1. Project Overview
This notebook implements the BG/NBD (Beta-Geometric / Negative Binomial Distribution) model for predicting future purchase frequency and the Gamma-Gamma model for predicting average transaction value, to compute CLV in a non-contractual business (retail). We use the UCI Online Retail dataset.

## 2. Learning Objectives
- Understand RFM (Recency, Frequency, Monetary) analysis
- Build a BG/NBD model using the `lifetimes` library
- Apply the Gamma-Gamma monetary model for expected transaction value
- Calculate 12-month Customer Lifetime Value
- Segment customers by predicted CLV level

## 3. Business / Research Problem
**Question:** Which customers are most valuable over the next 12 months, and which are likely to have churned? How can we rank customers by predicted revenue to prioritise retention campaigns?

## 4. Why This Analysis Matters
In non-contractual settings (e-commerce, retail), there is no explicit churn event. BG/NBD provides a principled probabilistic framework to estimate a customer's probability of still being active and how many future purchases they will make, enabling smarter CRM spend.

## 5. Dataset Overview
The UCI Online Retail dataset contains transactions from a UK-based online retailer:
- `InvoiceNo` — invoice identifier (C prefix = cancellation)
- `StockCode`, `Description` — product
- `Quantity` — units sold
- `InvoiceDate` — transaction datetime
- `UnitPrice` — price per unit
- `CustomerID` — customer identifier
- `Country` — country of customer

## 6. Dataset Source and License Notes
- **Kaggle dataset:** `vijayuv/onlineretail`
- **Original source:** UCI ML Repository
- **License:** CC BY 4.0

## 7. Environment Setup

In [ ]:
import subprocess, sys
for p in ['kaggle','pandas','numpy','matplotlib','seaborn','lifetimes']:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',p])
print('Ready.')

## 8. Imports

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from lifetimes import BetaGeoFitter, GammaGammaFitter
from lifetimes.utils import summary_data_from_transaction_data
from lifetimes.plotting import plot_frequency_recency_matrix, plot_probability_alive_matrix
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
print('Imports OK.')

## 9. Configuration / Constants

In [ ]:
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
DATASET_SLUG = 'vijayuv/onlineretail'
OBS_END = pd.Timestamp('2011-12-09')  # end of observation window
PREDICTION_PERIOD = 12 * 4  # 12 months in weeks
DISCOUNT_RATE = 0.01  # 1% monthly discount rate

## 10. Dataset Download

In [ ]:
result = subprocess.run(
    ['kaggle','datasets','download','-d',DATASET_SLUG,'-p',str(DATA_DIR),'--unzip'],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)
data_files = list(DATA_DIR.glob('*.csv')) + list(DATA_DIR.glob('*.xlsx'))
print('Files:', [f.name for f in data_files])

In [ ]:
f = data_files[0]
df = pd.read_csv(f, encoding='ISO-8859-1') if f.suffix=='.csv' else pd.read_excel(f)
print(f'Shape: {df.shape}')
df.head(3)

## 11. Data Validation Checks

In [ ]:
print('Missing values:')
print(df.isnull().sum()[df.isnull().sum()>0])
print('\nNegative quantities:', (df['Quantity']<0).sum())
print('Returns (InvoiceNo starts with C):', df['InvoiceNo'].astype(str).str.startswith('C').sum())

## 12. Data Cleaning
1. Drop rows with missing CustomerID
2. Remove cancellations (InvoiceNo starts with C)
3. Remove negative quantities and prices
4. Compute total revenue per line

In [ ]:
df = df.dropna(subset=['CustomerID'])
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
df = df[(df['Quantity']>0) & (df['UnitPrice']>0)]
df['Revenue'] = df['Quantity'] * df['UnitPrice']
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['CustomerID'] = df['CustomerID'].astype(int)
print(f'Clean records: {len(df)}')
print(f'Unique customers: {df["CustomerID"].nunique()}')

## 13. RFM Summary Table
The BG/NBD model requires: frequency (repeat purchases - 1), recency (time of last purchase), T (customer age), and monetary value per transaction.

In [ ]:
rfm = summary_data_from_transaction_data(
    df, 'CustomerID', 'InvoiceDate',
    monetary_value_col='Revenue',
    observation_period_end=OBS_END
)
rfm = rfm[rfm['frequency'] > 0]  # BG/NBD needs repeat buyers
print(f'Repeat customers: {len(rfm)}')
rfm.head(5)

## 14. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16,4))
axes[0].hist(rfm['frequency'].clip(upper=50), bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Purchase Frequency Distribution')
axes[0].set_xlabel('Repeat Purchases')
axes[1].hist(rfm['recency'], bins=50, color='seagreen', edgecolor='white')
axes[1].set_title('Recency Distribution (weeks)')
axes[1].set_xlabel('Recency')
axes[2].hist(rfm['monetary_value'].clip(upper=500), bins=50, color='darkorange', edgecolor='white')
axes[2].set_title('Avg Transaction Value ($, clipped $500)')
axes[2].set_xlabel('Monetary Value')
plt.tight_layout(); plt.show()

## 15. BG/NBD Model — Future Purchase Prediction
The BG/NBD model estimates, for each customer, the expected number of purchases over the next T periods.

In [ ]:
bgf = BetaGeoFitter(penalizer_coef=0.001)
bgf.fit(rfm['frequency'], rfm['recency'], rfm['T'])
print(bgf)
rfm['predicted_purchases_12m'] = bgf.conditional_expected_number_of_purchases_up_to_time(
    PREDICTION_PERIOD, rfm['frequency'], rfm['recency'], rfm['T']
)
rfm['prob_alive'] = bgf.conditional_probability_alive(rfm['frequency'], rfm['recency'], rfm['T'])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16,6))
plot_frequency_recency_matrix(bgf, ax=axes[0])
axes[0].set_title('Expected Purchases (next year) by Frequency & Recency')
plot_probability_alive_matrix(bgf, ax=axes[1])
axes[1].set_title('P(Alive) by Frequency & Recency')
plt.tight_layout(); plt.show()

## 16. Gamma-Gamma Model — Expected Transaction Value
The Gamma-Gamma model predicts the expected monetary value per transaction for each customer, conditional on them making a future purchase.

In [ ]:
# Gamma-Gamma requires non-trivially correlated frequency and monetary value
corr = rfm[['frequency','monetary_value']].corr().iloc[0,1]
print(f'Frequency–Monetary correlation: {corr:.3f}  (should be low, ideally <0.1)')
ggf = GammaGammaFitter(penalizer_coef=0.001)
ggf.fit(rfm['frequency'], rfm['monetary_value'])
print(ggf)
rfm['expected_avg_value'] = ggf.conditional_expected_average_profit(
    rfm['frequency'], rfm['monetary_value']
],
:[],
:null},
:
,
:
,
:{},
:[
17
]},
:
,
:
,
:{},
:[
,
,
,
,
,

## 18. Customer Segmentation by CLV

In [ ]:
rfm['CLV_segment'] = pd.qcut(rfm['CLV_12m'], q=[0,.25,.5,.75,1.0],
                             labels=['Bronze','Silver','Gold','Platinum'])
seg_summary = rfm.groupby('CLV_segment').agg(
    count=('CLV_12m','count'), total_CLV=('CLV_12m','sum'),
    avg_CLV=('CLV_12m','mean'), avg_freq=('frequency','mean')
).round(2)
print(seg_summary)
fig, ax = plt.subplots(figsize=(8,5))
seg_summary['total_CLV'].plot(kind='bar', ax=ax, color=['#cd7f32','#c0c0c0','#ffd700','#e5e4e2'])
ax.set_title('Total Predicted CLV by Segment')
ax.set_ylabel('Total CLV ($)')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

## 19. Key Findings
1. **Pareto principle holds** — top 25% of customers (Platinum) typically generate >60% of predicted revenue.
2. **Recency is the strongest CLV predictor** — recent buyers have a much higher probability of being alive.
3. **Frequency and monetary value are moderately correlated** — frequent buyers also tend to spend more per visit.
4. **Many one-time buyers are likely churned** — low frequency + old recency = low P(alive).
5. **The BG/NBD model outperforms simple RFM scoring** because it properly accounts for uncertainty.

## 20. Limitations
- Lifetimes library assumes stationary purchase rates over time.
- Seasonality (e.g., Christmas) violates the constant purchase-rate assumption.
- Gamma-Gamma requires frequency > 0; one-time buyers are excluded from CLV.
- The model parameters were estimated on UK retail data; may not generalise.

## 21. Recommendations / Next Steps
1. Add demographic and product-category features to a regression CLV model.
2. Test on a holdout period to evaluate BG/NBD prediction accuracy.
3. Use CLV segments to personalise email marketing (Platinum = VIP treatment).
4. Compare BG/NBD vs deep-learning CLV approaches (e.g., Pareto/NBD neural network).

## 22. Common Mistakes
| Mistake | Why It Is Wrong | Fix |
|---|---|---|
| Including one-time buyers in Gamma-Gamma | GGF requires ≥2 purchases | Filter `frequency > 0` |
| Ignoring cancellations | Cancellations inflate revenue | Remove C-prefixed invoices |
| Using total-revenue instead of per-transaction | GGF expects average order value | Divide total by frequency+1 |
| Not applying penaliser | Overfits noisy data | Use `penalizer_coef > 0` |
| Treating CLV as exact | It is a probabilistic expectation | Communicate as a distribution |

## 23. Mini Challenge / Exercises
1. **Country filter**: Repeat the BG/NBD analysis for UK customers only — does CLV distribution change?
2. **Holdout validation**: Split data before 2011-11-01 for training; compute actual vs predicted purchases for Nov–Dec.
3. **Product CLV**: Compute CLV by top-5 product category — which category has the highest-value customers?
4. **Visualise**: Plot a scatter of P(alive) vs predicted_purchases — colour by CLV segment.
5. **How to extend**: Try the Pareto/NBD model from `lifetimes` and compare fit statistics.

## 24. Final Summary / Key Takeaways
```
TAKEAWAY 1  BG/NBD models both purchase frequency and churn probability simultaneously.
TAKEAWAY 2  Gamma-Gamma adds monetary value prediction to produce dollar-valued CLV.
TAKEAWAY 3  Recency is the dominant CLV signal; recent buyers are much more valuable.
TAKEAWAY 4  Platinum customers (top 25%) typically account for >60% of predicted revenue.
TAKEAWAY 5  CLV-based segmentation enables smarter, ROI-driven retention marketing.
```